# Assessing Bicycle Lane Quality Using Smartphone Sensor Data

This notebook contains the complete data analysis pipeline for
classifying bicycle-lane sections as Smooth or Bumpy using
smartphone sensor data.

The analysis includes data loading, preprocessing, feature
extraction, feature selection, supervised and unsupervised
learning, model evaluation, and deployment.

> **Group members:**
>
> | Name | Student number |
> |---|---|
> | Himanshu Khartade | 2431459 |
> | *fill in* | *fill in* |
> | *fill in* | *fill in* |
>
> Participants are referred to as **P1, P2, P3** everywhere else (data files, plots, text) to keep the data anonymous.

# Environment and Reproducibility
The Python environment and package versions are recorded below to
support reproducibility.

In [1]:
import sys

import pandas as pd
import numpy as np
import sklearn
import matplotlib
import matplotlib.pyplot as plt

print("Python version:", sys.version)
print("pandas:", pd.__version__)
print("numpy:", np.__version__)
print("scikit-learn:", sklearn.__version__)
print("matplotlib:", matplotlib.__version__)

Python version: 3.13.5 | packaged by Anaconda, Inc. | (main, Jun 12 2025, 16:37:03) [MSC v.1929 64 bit (AMD64)]
pandas: 2.2.3
numpy: 2.3.5
scikit-learn: 1.9.0
matplotlib: 3.10.0


## 1. Study Design & Data Collection

**Aim** Build a tool that classifies bike-lane surface quality — *smooth* or *bumpy* — from the motion sensors of a
smartphone.

**Participants** 3 group members (P1-P3), all course participants who can bike and have no condition that prevents
cycling. Every participant recorded one smooth and one bumpy ride.

**Sensors & settings** Sensor Logger app; accelerometer, gyroscope and gravity sensor only (all other sensors
off); default 100 Hz; *Standardisation* option on so iOS and Android exports are comparable.

**Procedure** Start recording → put the phone on the basket → ride the lane continuously → stop → take
the phone out → stop recording. Lane choice followed the protocol: *smooth* = a typical red cycling lane without imperfections;
*bumpy* = a commonly used lane with noticeable but not dangerous imperfections (small potholes, cracks, uneven surface).

**Data management**
- `data/raw/` — untouched exports, one folder per participant and condition, consistent names, anonymized (P1-P3).
- `data/processed/` — everything the notebook derives (merged + trimmed recordings, window features), stored separately.

In [ ]:
Collection_info = {'P1':{'Phone':'iphone 15', 'placement':'Vertical on bicycle basket'},
                   'P2':{'Phone':'iphone 13', 'placement':'Vertical on bicycle basket'},
                   'P3':{'Phone':'iphone 17 pro max', 'placement':'Vertical on bicycle basket'}}

Route_info = {'P1_smooth': 'Near Geldropsedijk bus stop','P1_bumpy': 'Eisenhowerlaan',
    'P2_smooth': 'Near Geldropsedijk bus stop','P2_bumpy': '',
    'P3_smooth': '','P3_bumpy': ''}

In [3]:
def load_and_merge(acc_path, grav_path, gyro_path):
    """
    Load accelerometer, gravity, and gyroscope data and combine
    them using their timestamps.
    """

    acc = pd.read_csv(acc_path).rename(
        columns={'x': 'acc_x', 'y': 'acc_y', 'z': 'acc_z'}
    )

    grav = pd.read_csv(grav_path).rename(
        columns={'x': 'grav_x', 'y': 'grav_y', 'z': 'grav_z'}
    )

    gyro = pd.read_csv(gyro_path).rename(
        columns={'x': 'gyro_x', 'y': 'gyro_y', 'z': 'gyro_z'}
    )

    # Check that all recordings have the same number of observations
    if not (len(acc) == len(grav) == len(gyro)):
        raise ValueError("Sensor files have different numbers of rows.")

    # Check timestamp alignment
    if not acc['time'].equals(grav['time']):
        raise ValueError("Accelerometer and gravity timestamps are not aligned.")

    if not acc['time'].equals(gyro['time']):
        raise ValueError("Accelerometer and gyroscope timestamps are not aligned.")

    # Combine the sensor measurements
    df = acc[
        ['time', 'seconds_elapsed', 'acc_x', 'acc_y', 'acc_z']
    ].copy()

    df[['grav_x', 'grav_y', 'grav_z']] = grav[
        ['grav_x', 'grav_y', 'grav_z']
    ]

    df[['gyro_x', 'gyro_y', 'gyro_z']] = gyro[
        ['gyro_x', 'gyro_y', 'gyro_z']
    ]

    return df


In [8]:
# Define the location of each participant's recordings

recordings = {
    'P1_smooth': {
        'label': 'smooth',
        'folder': 'data_raw/Smooth/P1'
    },
    'P1_bumpy': {
        'label': 'bumpy',
        'folder': 'data_raw/Bumpy/P1'
    },
    'P2_smooth': {
        'label': 'smooth',
        'folder': 'data_raw/Smooth/P2'
    },
    'P2_bumpy': {
        'label': 'bumpy',
        'folder': 'data_raw/Bumpy/P2'
    },
    'P3_smooth': {
        'label': 'smooth',
        'folder': 'data_raw/Smooth/P3'
    },
    'P3_bumpy': {
        'label': 'bumpy',
        'folder': 'data_raw/Bumpy/P3'
    }
}

In [ ]:
# Dictionary to store the merged data for each recording
merge_recordings = {}

for name, recording in recordings.items():

    folder = recording['folder']

# Load and merge accelerometer, gravity, and gyroscope data
    df = load_and_merge(
        f"{folder}/Accelerometer.csv",
        f"{folder}/Gravity.csv",
        f"{folder}/Gyroscope.csv")

# Store the merged data using the recording name
    merge_recordings[name] = df
    
# Display the number of samples and recording duration
    print(
        f"{name}: "
        f"{len(df)} rows, "
        f"{df['seconds_elapsed'].max():.1f} seconds"
    )

P1_smooth: 7719 rows, 77.3 seconds
P1_bumpy: 18010 rows, 180.4 seconds
P2_smooth: 12576 rows, 126.8 seconds
P2_bumpy: 19553 rows, 197.2 seconds
P3_smooth: 4763 rows, 47.8 seconds
P3_bumpy: 7583 rows, 76.2 seconds


## 3. Data Quality Assessment & Exploratory Data Analysis

Before any modeling: is the data complete and regular, and do the rides look like what they are supposed to be?

Per recording: sample count, duration, effective sampling rate, largest gap between samples, missing values and duplicated
timestamps. (Timestamp alignment of the three sensors was already asserted while loading.)

In [21]:
def recording_quality(key, df):
    t = df['seconds_elapsed'].values

    participant = key.split('_')[0]

    # Return a dictionary containing metadata and calculated quality metrics
    return {
        'recording': key,
    
        'participant':participant,

        # Look up device and placement metadata from the external Collection_info dictionary
        'phone': Collection_info[participant]['Phone'],   
        'placement': Collection_info[participant]['placement'],

        # Look up route metadata from the external Route_info dictionary
        'route': Route_info[key],  

        'rows': len(df),
        # Calculate total duration in seconds (last timestamp minus first timestamp)
        'duration_s': round(t[-1] - t[0], 1),

        # Calculate effective sampling frequency (Hz): (Total intervals) / (Total duration)
        'effective_hz': round(
            (len(t) - 1) / (t[-1] - t[0]), 1),

        # Count the total number of missing (NaN) values across all columns in the DataFrame
        'missing_values': int(
            df.isna().sum().sum()),

        # Count how many timestamps in the 'time' column are exact duplicates
        'duplicate_timestamps': int(
            df['time'].duplicated().sum()),
    }

# Create a summary DataFrame aggregating the quality metrics for all recordings
quality = pd.DataFrame(
    [
        recording_quality(name, df)
        for name, df in merge_recordings.items()
    ]
).set_index('recording')

display(quality)
t_dur = round(quality['duration_s'].sum() / 60,1)
print(f'Total duration: {t_dur} mins')

,participant,phone,placement,route,rows,duration_s,effective_hz,missing_values,duplicate_timestamps
recording,,,,,,,,,
P1_smooth,P1,iphone 15,Vertical on bicycle basket,Near Geldropsedijk bus stop,7719,77.3,99.9,0,0
P1_bumpy,P1,iphone 15,Vertical on bicycle basket,Eisenhowerlaan,18010,180.3,99.9,0,0
P2_smooth,P2,iphone 13,Vertical on bicycle basket,,12576,126.7,99.2,0,0
P2_bumpy,P2,iphone 13,Vertical on bicycle basket,,19553,197.1,99.2,0,0
P3_smooth,P3,iphone 17 pro max,Vertical on bicycle basket,,4763,47.8,99.6,0,0
P3_bumpy,P3,iphone 17 pro max,Vertical on bicycle basket,,7583,76.1,99.6,0,0


Total duration: 11.8 mins


# Data Preprocessing